# Differentiable minilink
## Write `f` once, get every gradient

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/learn/intro/showcase_jax.ipynb)

**The payoff.** A minilink model is a pure function of its arguments:

$$\dot{x} = f(x, u, t; p) \qquad y = h(x, u, t; p)$$

Because nothing is hidden on the object, automatic differentiation can take the
derivative of `f`, or of a whole simulation built from `f`, with respect to **any**
argument: the state `x`, the input `u`, the parameters `p`, or an input sequence.
Exactly, to machine precision, and cheaply. That one capability is the engine behind
linearization, control design, identification, tuning, trajectory optimization and
model predictive control. You write the physics once; the gradients follow.

Every block is a `System` — a plant, a PID, a neural law, a wired diagram. Because
`f` traces under JAX, the same object answers a classical frequency question and a
learning question: `plot_bode` on an impedance loop, `linearize` through a network,
one residual. Section 10 shows that on this pendulum; the UR5 lab
([ur5_impedance_rl](../teaching/ur5_impedance_rl.ipynb)) puts both on one arm.

This notebook is the *why* and the *what you get*. The compile API itself
(`compile(backend)`, evaluator tiers, where tools plug in) is the subject of
[07_compile](07_compile.ipynb).

**Contents**

1. [The stateless contract](#1.-The-stateless-contract)
2. [A textbook model, written once](#2.-A-textbook-model,-written-once)
3. [What JAX tracing is](#3.-What-JAX-tracing-is)
4. [∂f/∂x and ∂f/∂u: linearization](#4.-∂f/∂x-and-∂f/∂u:-linearization)
5. [∂f/∂p: sensitivity to the physics](#5.-∂f/∂p:-sensitivity-to-the-physics)
6. [Identification by gradient descent](#6.-Identification-by-gradient-descent)
7. [Differentiate through a rollout](#7.-Differentiate-through-a-rollout): an input sequence, then a physical parameter
8. [A family of parameters in one call](#8.-A-family-of-parameters-in-one-call)
9. [Speed](#9.-Speed)
10. [Everything is a System](#10.-Everything-is-a-System): Bode and a neural law on the same contract
11. [Recap](#11.-Recap)

In [ ]:
# Local conda: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

## 1. The stateless contract

Every block, plant and controller extends a base `System` whose dynamical contract is
the two equations above. The convention, stated in `minilink/core/system.py`, is that an
overridden `f` or `h` behaves as a function of `(x, u, t, params)` only: no mutation of
`self`, no integrator buffer, no cached force.

A simulator that instead steps the model by mutating internal fields is convenient
until you want to do anything other than run it forward: results start to depend on
call order, parallel evaluation aliases, and the model is opaque to a differentiation
engine. Keeping `f` pure buys three things at once:

- **Reproducibility and composability.** `f(x, u, t, p)` always returns the same answer
  for the same arguments, so systems compose into diagrams and a trajectory is just
  `x0` plus a sequence of inputs.
- **Batching.** A pure function evaluates over a whole batch of states, inputs or
  parameters at once (`jax.vmap`).
- **Differentiability.** A pure function can be traced and differentiated. The rest of
  this notebook is about that.

## 2. A textbook model, written once

The catalog pendulum is a `MechanicalSystem`: `f` assembles the generalized
acceleration from the inertia, Coriolis, gravity and damping terms, and every physical
constant lives in `params`. We set the hub inertia to zero so the rod length enters the
equations in the classic form, $m \ell^2 \ddot\theta = \tau - m g \ell \sin\theta - d\,\dot\theta$.

In [ ]:
import numpy as np

from minilink import Pendulum

pendulum = Pendulum()
pendulum.params["I"] = 0.0  # point mass on a massless rod
pendulum.params["d"] = 0.2  # viscous damping [N m s]
print(pendulum.params)

x = np.array([0.5, 1.0])  # theta [rad], dtheta [rad/s]
u = np.array([0.0])  # torque [N m]
print("dx/dt =", pendulum.f(x, u))  # a plain NumPy call

For your own plant the one idiom is `array_module(x)`: it returns `numpy` for NumPy
arrays and `jax.numpy` for JAX arrays, so the same `f` serves both backends with no
second implementation and no `if jax:` branch. Every catalog plant is written this way.

In [ ]:
from minilink import DynamicSystem
from minilink.core.backends import array_module


class MyPendulum(DynamicSystem):
    def __init__(self):
        super().__init__(n=2, input_dim=1, output_dim=2)
        self.params = {"m": 1.0, "l": 1.0, "gravity": 9.81, "d": 0.2}

    def f(self, x, u, t=0, params=None):
        p = self.params if params is None else params
        xp = array_module(x)  # numpy or jax.numpy, chosen by the arguments
        q, dq = x[0], x[1]
        ddq = (u[0] - p["m"] * p["gravity"] * p["l"] * xp.sin(q) - p["d"] * dq) / (p["m"] * p["l"] ** 2)
        return xp.array([dq, ddq])


print("same physics, custom class:", MyPendulum().f(x, u))
print("traces under JAX          :", MyPendulum().compile(backend="jax").f(x, u, 0.0))

## 3. What JAX tracing is

JAX does not differentiate Python source text. It **traces**: it runs the function once
on abstract placeholder values and records the sequence of array operations into a
small graph (a `jaxpr`). That graph can then be compiled (`jit`), differentiated
(`grad`, `jacfwd`, `jacrev`) or mapped over a batch (`vmap`).

Tracing only works if the function is pure. If `f` mutated `self`, read a clock, or
branched on the *value* of a tracer, the recorded graph would be wrong or tracing would
fail. That is exactly why the contract of section 1 matters: *stateless `f` is the
precondition for differentiable `f`.*

`compile(backend="jax")` traces `f` and `h` once and returns an evaluator with
JIT-compiled primitives. JAX evaluators use 64-bit floats, so the derivatives below
are computed in double precision.

In [ ]:
import jax
import jax.numpy as jnp

evaluator = pendulum.compile(backend="jax")  # traces f/h once, JIT-compiles to XLA

x_op = jnp.array([0.5, 1.0])  # one operating point reused below
u_op = jnp.array([0.0])
t_op = jnp.array(0.0)
print("dx/dt =", evaluator.f(x_op, u_op, t_op))

## 4. ∂f/∂x and ∂f/∂u: linearization

The Jacobians of `f` with respect to the state and the input are the **A** and **B**
matrices of the local linear model $\dot{\delta x} = A\,\delta x + B\,\delta u$. The
eigenvalues of `A` report local stability; the pair $(A, B)$ drives controllability
tests and LQR design (`lqr_at_operating_point` is built on exactly these). Both are the
same one-liner, `jacfwd` of `f`, differing only in which argument varies.

In [ ]:
A = np.asarray(jax.jacfwd(lambda x: evaluator.f(x, u_op, t_op))(x_op))
B = np.asarray(jax.jacfwd(lambda u: evaluator.f(x_op, u, t_op))(u_op))
print("A = df/dx =\n", A)
print("B = df/du =\n", B)
print("local eigenvalues:", np.linalg.eigvals(A))
print("rank of [B, A B] =", np.linalg.matrix_rank(np.column_stack([B, A @ B])), "(2 => controllable)")

minilink exposes this as `plant.jacobian("f", "x")`, read it as ∂f/∂x, with exact
autodiff when the plant traces (`method="jax"`, the default under `"auto"`) or central
finite differences (`method="fd"`). They agree; the autodiff version is exact, while
finite differences trade accuracy against step size.

In [ ]:
A_jax = pendulum.jacobian("f", "x", np.asarray(x_op), np.asarray(u_op), method="jax")
A_fd = pendulum.jacobian("f", "x", np.asarray(x_op), np.asarray(u_op), method="fd")
print("A (jax autodiff) =\n", A_jax)
print("A (finite diff)  =\n", A_fd)
print("max |jax - fd| =", np.max(np.abs(A_jax - A_fd)))

## 5. ∂f/∂p: sensitivity to the physics

This is the direction hidden state would make impossible. Because the parameters are an
explicit argument, `f` differentiates with respect to **the physics itself**:
`pendulum.jacobian("f", "params")` returns ∂f/∂p as a dict mirroring `params`, one
sensitivity vector per parameter. Below, how the angular acceleration responds to each
constant, cross-checked against finite differences.

In [ ]:
exact = pendulum.jacobian("f", "params", np.asarray(x_op), np.asarray(u_op))
fd = pendulum.jacobian("f", "params", np.asarray(x_op), np.asarray(u_op), method="fd")

print(f"{'param':>10} {'autodiff d(ddq)/dp':>20} {'finite diff':>14}")
for key in pendulum.params:
    print(f"{key:>10} {float(exact[key][1]):>20.6f} {float(fd[key][1]):>14.6f}")

## 6. Identification by gradient descent

Sensitivity to parameters makes **system identification** a gradient problem. Suppose
we logged states and inputs but do not know the true gravity and damping. Minimize the
equation-error loss

$$\mathcal{L}(\theta) = \frac{1}{N}\sum_k \big\| f(x_k, u_k, t_k;\, \theta) - \dot{x}_k \big\|^2$$

by gradient descent, with $\nabla_\theta \mathcal{L}$ produced by `jax.value_and_grad`
and `f` evaluated over the whole dataset at once by `jax.vmap`. The data comes from the
evaluator's own RK4 rollout (`rk4_integrate_zoh`, the jit tier); the loss calls the
parametric trace tier (`f_trace_p`), the same `f` with `params` as an argument.

In [ ]:
dt = 0.02
n_samples = 400
ts = dt * jnp.arange(n_samples)
us = 0.5 * jnp.sin(0.8 * ts).reshape(-1, 1)

# 1. "Measured" data from the true pendulum: one rollout, then f at every sample
xs = evaluator.rk4_integrate_zoh(jnp.array([0.5, 0.0]), us, 0.0, dt)[:-1]
dxs = jax.vmap(evaluator.f_trace, in_axes=(0, 0, 0))(xs, us, ts)

# 2. Identify gravity and damping by gradient descent on the equation error
def loss(theta):
    p = dict(pendulum.params, gravity=theta[0], d=theta[1])
    dx_hat = jax.vmap(evaluator.f_trace_p, in_axes=(0, 0, 0, None))(xs, us, ts, p)
    return jnp.mean((dx_hat - dxs) ** 2)

loss_and_grad = jax.jit(jax.value_and_grad(loss))
theta = jnp.array([7.0, 1.0])  # wrong initial guess for (gravity, damping)
print(f"{'iter':>6} {'loss':>12} {'gravity':>9} {'damping':>9}")
for i in range(151):
    value, grad = loss_and_grad(theta)
    if i % 25 == 0:
        print(f"{i:>6} {float(value):>12.3e} {float(theta[0]):>9.4f} {float(theta[1]):>9.4f}")
    theta = theta - 2.0 * grad
print(f"identified gravity = {float(theta[0]):.4f}  (true 9.81)")
print(f"identified damping = {float(theta[1]):.4f}  (true 0.20)")

## 7. Differentiate through a rollout

The purity that lets us differentiate $f$ lets us differentiate a whole **trajectory**
built from $f$. One RK4 step is a map $x_{k+1} = \Phi(x_k, u_k; p)$, and an $N$-step
rollout is its composition,

$$x_N(U, p) = \Phi(\cdots\Phi(\Phi(x_0, u_0; p), u_1; p)\cdots, u_{N-1}; p), \qquad U = (u_0, \ldots, u_{N-1}).$$

Any scalar cost $J(U, p)$ of that rollout is a composition of pure functions, so
automatic differentiation returns $\nabla_U J$ or $\nabla_p J$ through every step,
with no hand-derived adjoint. The evaluator's `rk4_integrate_zoh_trace` (and its
parametric twin `rk4_integrate_zoh_trace_p`) is that composition, ready to
differentiate.

The same goal, put the final angle on a target $\theta^\star$, is solved twice: by
choosing the torque sequence $U$ with the physics fixed, then by choosing one physical
parameter with $u \equiv 0$.

### 7a. An input sequence

Single shooting: hold $p$ fixed, treat $U$ as the decision variable, and minimize
$J(U) = (\theta_N(U) - \theta^\star)^2 + \varepsilon \sum_k u_k^2$. Gradient descent on
$U$ needs $\partial x_N / \partial U$, the sensitivity of the rollout to each torque
sample, which `jax.value_and_grad` produces. This is the conceptual core of trajectory
optimization and of the plan inside MPC; `TrajectoryOptimizationPlanner` does the
transcription with these gradients for you.

In [ ]:
import matplotlib.pyplot as plt

H1, dt1 = 40, 0.05
x_down = jnp.array([0.0, 0.0])  # hanging down, at rest
theta_target_1 = 2.0  # final angle [rad]


def cost_u(U):
    xs = evaluator.rk4_integrate_zoh_trace(x_down, U, 0.0, dt1)  # (H1 + 1, 2)
    return (xs[-1, 0] - theta_target_1) ** 2 + 1e-4 * jnp.sum(U**2)


value_and_grad_u = jax.jit(jax.value_and_grad(cost_u))
U = jnp.zeros((H1, 1))
for _ in range(1500):
    _, g = value_and_grad_u(U)
    U = U - 0.6 * g  # gradient descent on the whole input sequence

traj_u = np.asarray(evaluator.rk4_integrate_zoh_trace(x_down, U, 0.0, dt1))
t1 = dt1 * np.arange(H1 + 1)
print(f"J(U*) = {float(cost_u(U)):.3e};  final angle {traj_u[-1, 0]:.3f} rad (target {theta_target_1})")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(t1, traj_u[:, 0], label=r"$\theta(t)$")
ax[0].axhline(theta_target_1, ls="--", color="k", lw=0.8, label=r"$\theta^\star$")
ax[0].set_xlabel("time [s]"); ax[0].set_ylabel("theta [rad]"); ax[0].legend()
ax[1].step(t1[:-1], np.asarray(U)[:, 0], where="post", color="C1", label=r"$u^\star_k$")
ax[1].set_xlabel("time [s]"); ax[1].set_ylabel("torque [N m]"); ax[1].legend()
fig.tight_layout()
plt.show()

### 7b. A physical parameter

Flip the roles: no torque, release the pendulum from rest, and ask which rod length
$\ell$ makes the angle at the final time equal a target,
$J(\ell) = (\theta_N(\ell) - \theta^\star)^2$. The gradient $dJ/d\ell$ is the same
chain rule through the same RK4 composition, now with respect to the physics rather
than the control. Only the argument of `value_and_grad` changes: that is design or
tuning rather than planning.

In [ ]:
H2, dt2 = 50, 0.01  # 0.5 s of free swing
theta0 = 1.0  # released from 1.0 rad, at rest
theta_target_2 = 0.3  # angle wanted at the final time
U_zero = jnp.zeros((H2, 1))


def theta_final(length):
    p = dict(pendulum.params, l=length)  # only the length varies
    xs = evaluator.rk4_integrate_zoh_trace_p(jnp.array([theta0, 0.0]), U_zero, 0.0, dt2, p)
    return xs[-1, 0]


def cost_l(length):
    return (theta_final(length) - theta_target_2) ** 2


value_and_grad_l = jax.jit(jax.value_and_grad(cost_l))
length = 1.0
for _ in range(300):
    _, g = value_and_grad_l(length)
    length = length - 2.0 * g  # gradient descent on the rod length

print(f"rod length 1.000 m -> {float(length):.3f} m")
print(f"final angle {float(theta_final(1.0)):.3f} rad -> {float(theta_final(length)):.3f} rad (target {theta_target_2})")

ells = np.linspace(0.5, 2.0, 60)
Js = np.array([float(cost_l(e)) for e in ells])
plt.plot(ells, Js, color="C1")
plt.axvline(float(length), ls="--", color="k", label=r"$\ell^\star$")
plt.xlabel(r"rod length $\ell$ [m]"); plt.ylabel(r"$J(\ell)$"); plt.legend()
plt.show()

## 8. A family of parameters in one call

Batching is the other thing purity buys. `rollout_batch` takes a batch of initial states
and, optionally, a `params` dict whose leaves carry one value per rollout: below, seven
rod lengths in one call. Plotted against the dimensionless time $t\sqrt{g/\ell}$, the
seven swings collapse onto one curve, which is the Buckingham-π statement that the
undamped pendulum has a single dimensionless period.

In [ ]:
lengths = np.linspace(0.5, 2.0, 7)
family = dict(pendulum.params, l=lengths, d=0.0)  # undamped, one length per rollout
x0s = np.tile(np.array([1.0, 0.0]), (len(lengths), 1))

DT, STEPS = 0.005, 1200
xs = np.asarray(evaluator.rollout_batch(x0s, n_steps=STEPS, dt=DT, params=family))  # (7, 1201, 2)
t = DT * np.arange(STEPS + 1)

fig, (ax_t, ax_pi) = plt.subplots(1, 2, figsize=(10, 3.4))
for x_l, ell in zip(xs, lengths):
    ax_t.plot(t, x_l[:, 0], label=f"l = {ell:.2f} m")
    ax_pi.plot(t * np.sqrt(pendulum.params["gravity"] / ell), x_l[:, 0])
ax_t.set_xlabel("t [s]"); ax_t.set_ylabel("theta [rad]"); ax_t.legend(fontsize=7)
ax_pi.set_xlabel(r"$t\,\sqrt{g/\ell}$  [-]"); ax_pi.set_title("same swing in dimensionless time")
fig.tight_layout()
plt.show()

The same works one level down: `jax.vmap` over the parametric trace tier evaluates `f`
for a whole family of parameters at once, which is the building block for policies that
transfer across physical parameters.

In [ ]:
ddq_family = jax.vmap(lambda ell: evaluator.f_trace_p(x_op, u_op, t_op, dict(pendulum.params, l=ell)))(jnp.asarray(lengths))
print("d(dtheta)/dt at x_op for each length:", np.round(np.asarray(ddq_family)[:, 1], 3))

## 9. Speed

A compiled batch is also fast. The cell times the same work two ways: 1000 rollouts of
1000 RK4 steps as one `rollout_batch` call on the JAX evaluator, and one rollout stepped
in a Python loop on the NumPy evaluator, scaled by 1000. The numbers depend on the
machine; the ratio does not depend much on it.

In [ ]:
import time

N, STEPS, DT = 1000, 1000, 0.005
x0s = np.column_stack([np.linspace(-3.0, 3.0, N), np.zeros(N)])

_ = np.asarray(evaluator.rollout_batch(x0s, n_steps=STEPS, dt=DT))  # first call compiles
t0 = time.perf_counter()
_ = np.asarray(evaluator.rollout_batch(x0s, n_steps=STEPS, dt=DT))
t_batch = time.perf_counter() - t0

ev_np = pendulum.compile(backend="numpy")
t0 = time.perf_counter()
x_k = x0s[0]
for k in range(STEPS):
    x_k = ev_np.rk4_step(x_k, np.zeros(1), k * DT, DT)
t_one = time.perf_counter() - t0

print(f"{N} rollouts x {STEPS} RK4 steps, compiled batch : {1e3 * t_batch:8.1f} ms")
print(f"1 rollout in a Python loop, x{N}                 : {1e3 * t_one * N:8.0f} ms")
print(f"ratio                                            : {t_one * N / t_batch:8.0f} x")

## 10. Everything is a System

A plant is a `System`. A controller is a `System`. `ctl @ plant` is a `System`. The
neural law a planner returns is a `System`. JAX tracing does not care which: the
closed-loop residual is one $f$, so the frequency tools and the Jacobians apply to a
classical loop and to a learned law.

The same pendulum as above. First an impedance loop — Bode of $\theta/r$ at the
hanging equilibrium, the classical channel. Then an untrained
`NeuralPolicyController` on the same plant: `linearize` differentiates through the
network and the physics in one trace. The UR5 lab
([ur5_impedance_rl](../teaching/ur5_impedance_rl.ipynb)) puts both on one arm: a
learned set-point law over joint impedance, then the sensitivity Bode $p_z/f_z$ of
that diagram.

In [ ]:
from minilink import ImpedanceController
from minilink.control import NeuralPolicyController

# Classical loop: Bode applies because the diagram is a System
loop = ImpedanceController() @ pendulum
loop.name = "Impedance loop"
x_bar = np.array([0.0, 0.0])
print("impedance-loop poles:", np.round(np.linalg.eigvals(loop.linearize(x_bar).A()), 2))
loop.plot_bode(x_bar, margins=False)

# A neural law is the same kind of block: linearize traces through the network
pendulum.inputs["u"].lower_bound = np.array([-4.0])
pendulum.inputs["u"].upper_bound = np.array([4.0])
rl_ctl = NeuralPolicyController(pendulum, hidden=(8, 8))
learned = rl_ctl @ pendulum
A = learned.linearize(x_bar).A()
print("A through the network:\n", np.round(A, 3))
print("neural-loop poles:", np.round(np.linalg.eigvals(A), 2))

## 11. Recap

Writing `f` once is only half the story. `compile()` lowers a leaf or a wired diagram
into an **evaluator**: flat primitives (`f`, `rk4_step`, `rk4_integrate_zoh`,
`rollout_batch`, …) that simulators, planners and MPC call thousands of times per
second, on NumPy or JAX, with the trace tier (`f_trace`, `rk4_integrate_zoh_trace_p`, …)
kept for your own `jit` and `grad`. Diagrams flatten, so nested `+` / `>>` / `@` become
one fast residual with no Python wiring in the inner loop.

| Idea | What it unlocks |
| --- | --- |
| Pure `f` / `h`, no shadow state | One physics definition for simulation, autodiff, identification, planning |
| JAX tracing | Exact ∂f/∂x, ∂f/∂u, ∂f/∂p |
| ∂f/∂x, ∂f/∂u | A and B matrices: linearization, LQR, stability |
| ∂f/∂p | Identification, tuning, sensitivity |
| Differentiate through a rollout | $\nabla_U J$ (trajectory optimization, MPC) and $\nabla_p J$ (design) without hand adjoints |
| `rollout_batch`, `vmap` | A family of parameters or initial conditions in one call |
| Compile → evaluator | Fast primitives and autodiff on the same model |
| Every block is a `System` | Bode, poles and autodiff on a classical loop and on a neural law |

**See also:** [07_compile](07_compile.ipynb) (the API), [showcase_minilink](showcase_minilink.ipynb)
(the tool ladder), [cartpole_rollout_gradients](../teaching/cartpole_rollout_gradients.ipynb)
(single shooting on a cart-pole, co-design of inputs and parameters),
[ur5_impedance_rl](../teaching/ur5_impedance_rl.ipynb)
(impedance + a learned law + a sensitivity Bode on one arm).